In [1]:
!wget https://www.dropbox.com/s/pdhwlpi2yeie0ol/movie-reviews-dataset.zip

--2026-07-18 19:59:54--  https://www.dropbox.com/s/pdhwlpi2yeie0ol/movie-reviews-dataset.zip
Resolving www.dropbox.com (www.dropbox.com)... 162.125.81.18, 2620:100:6030:18::a27d:5012
Connecting to www.dropbox.com (www.dropbox.com)|162.125.81.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/4r8fb499vfpyrw44fgftj/movie-reviews-dataset.zip?rlkey=79qfzf6683udd2ehdii38y7wt [following]
--2026-07-18 19:59:55--  https://www.dropbox.com/scl/fi/4r8fb499vfpyrw44fgftj/movie-reviews-dataset.zip?rlkey=79qfzf6683udd2ehdii38y7wt
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uc5753163af2fe82d23b48c45504.dl.dropboxusercontent.com/cd/0/inline/DEjr1Ohhf7k4P83KKet6myhgTeKjfeHVs1gXt0Gdfl05c3Cu3NUKx7GLI0oXyNOQq9LArSvgUn-sAHelsOVcQHORYDrPD0VnG6rzdfm7kel00Pl1PuQaeMQRpX9_4PNutpi1mhMdd5QWHAd_I2YhGh8i/file# [following]
--2026-07-18 19:59:55--  https://uc5753163af2fe82d23b48c455

In [2]:
!unzip -q "/content/movie-reviews-dataset.zip"

In [3]:
from tensorflow.keras.preprocessing import text_dataset_from_directory
from tensorflow.strings import regex_replace
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.models import Sequential
from tensorflow.keras import Input
from tensorflow.keras.layers import Dense, LSTM, Embedding, Dropout , Bidirectional

In [4]:
def prepareData(dir):
  data = text_dataset_from_directory(dir)
  return data.map(
    lambda text, label: (regex_replace(text, '<br />', ' '), label),
  )

In [5]:
train_data = prepareData('movie-reviews-dataset/train')
test_data = prepareData('movie-reviews-dataset/test')

for text_batch, label_batch in train_data.take(1):
  print(text_batch.numpy()[0])
  print(label_batch.numpy()[0])

Found 25000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.
b"The Tooth Fairy is set in a small town somewhere in Northern California where Peter Campbell (Lochlyn Munro) has brought a farming property which he is renovating & planning to turn into a holiday inn, he is joined by his girlfriend Darcy Wagner (Chandra West) & her young 12 year old daughter Pamela (Nicole Munoz) who arrive to help for the weekend. While exploring the property Pamela meets another young girl named Emma (Jianna Ballard) who warns her that evil lurks within her new home, she tells a tale of an evil old witch known as the Tooth Fairy who takes baby teeth from children & then kills them. Pamela is worried & becomes even more so when she falls off her bike & her last baby tooth falls out, it's not long before the evil ghost of the Tooth Fairy has her eyes on Pamela's tooth & just for kicks she also decides to kill anyone she comes across...  Directed by Chuck Bowman I thought The Tooth Fa

In [6]:
model = Sequential()

In [7]:
model.add(Input(shape=(), dtype="string"))

In [8]:
max_tokens = 1000
max_len = 100
vectorize_layer = TextVectorization(
  max_tokens=max_tokens,
  output_mode="int",
  output_sequence_length=max_len,
)

In [9]:
train_texts = train_data.map(lambda text, label: text)
vectorize_layer.adapt(train_texts)

model.add(vectorize_layer)

In [10]:
model.add(Embedding(max_tokens + 1, 128))

model.add(Bidirectional(LSTM(64)))
model.add(Dense(64, activation="relu"))
model.add(Dense(1, activation="sigmoid"))

In [11]:
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

In [12]:
model.fit(train_data, epochs=10)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 16s 16ms/step - accuracy: 0.7383 - loss: 0.5148
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - accuracy: 0.8002 - loss: 0.4317
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - accuracy: 0.8167 - loss: 0.3974
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - accuracy: 0.8378 - loss: 0.3644
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - accuracy: 0.8515 - loss: 0.3325
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - accuracy: 0.8674 - loss: 0.3014
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - accuracy: 0.8806 - loss: 0.2727
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 20s 15ms/step - accuracy: 0.8933 - loss: 0.2439
Epoch 9/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - accuracy: 0.9046 - loss: 0.2162
Epoch 10/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 21s 15ms/step - accuracy: 0.9183 - loss: 0.1904


In [13]:
model.evaluate(test_data)

782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.7758 - loss: 0.8537


[0.853683352470398, 0.7757999897003174]

In [31]:
text = "naruto is a 3/10"

In [32]:
import tensorflow as tf
model.predict(tf.constant([text]))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step


array([[0.9543622]], dtype=float32)